<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
if os.path.basename(os.getcwd()) == "KE4MHQ":
    os.chdir("rome")
!ls


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution
import json
import time

from util.eval_greedy import eval_editing


import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)  # Python random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Ensure deterministic behavior
    torch.backends.cudnn.benchmark = False  # Disable auto-optimization

set_seed(42)

 baselines		      'hop1-Eval-[15]'		    hparams
'both-Eval-[5]-[10]'	      'hop1-Eval-[20]'		    LICENSE
'both-Eval-[5, 15]-[10, 20]'  'hop1-Eval-[5]'		    logs
'both-Eval-[5]-[5]'	      'hop1-Eval-[5, 10, 15, 20]'   notebooks
 CITATION.cff		       Hop1-Eval-5-10-15-20	    README.md
 data			      'hop2-Eval-[10]'		    results
 dsets			      'hop2-Eval-[15]'		    rome
 experiments		      'hop2-Eval-[20]'		    scripts
 globals.yml		      'hop2-Eval-[5]'		    util
'hop1-Eval-[10]'	      'hop2-Eval-[5, 10, 15, 20]'
 Hop1-Eval-10		       hop2-Eval-5-10-15-20


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
device = torch.device('cuda:0')
print(f"Using device: {device}")


Using device: cuda:0


In [4]:

# del model  # Deletes the model from memory
# torch.cuda.empty_cache()  # Clears unused memory from the GPU
# torch.cuda.ipc_collect()  # Helps reclaim unused memory


In [5]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [11]:
ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
layers_to_edit = [10]

json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}

with open(json_path_dict[MODEL_NAME], "r") as f:
    data = json.load(f)
data["layers"] = layers_to_edit
with open(json_path_dict[MODEL_NAME], "w") as f:
    json.dump(data, f, indent=2)



In [7]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",

#  Pipeline for testing multiple insertions on MQuake

In [8]:

ALG_NAME = "ROME-Multi" # "ROME-Multi"
MODEL_NAME = "EleutherAI/gpt-j-6B" # "gpt2-xl"
# layers_to_edit =[[5],[10]]
# layers_to_edit = [[5,10,15,20]]


json_path_dict = {
    "gpt2-xl":"hparams/ROME/gpt2-xl.json",
    "EleutherAI/gpt-j-6B":"hparams/ROME/EleutherAI_gpt-j-6B.json"
}



# edit_hop = "both" # choose one from ["hop1", "hop2", "both"]
# # ds_file = "dsets/single_edit_0-100.json"
# ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
# with open(ds_file, "r") as f:
#     mhq_ds = json.load(f)

# context used when testing the editted model
context_file = "dsets/rel-prompts.json"
with open(context_file, "r") as f:
    rel_prompts = json.load(f)




def test_multi_rome(layers_to_edit, edit_hop, continue_from=0):

    with open(json_path_dict[MODEL_NAME], "r") as f:
        data = json.load(f)
    data["layers"] = layers_to_edit
    with open(json_path_dict[MODEL_NAME], "w") as f:
        json.dump(data, f, indent=2)

    ds_file = "dsets/ds_classification/"+edit_hop+"_edits.json"
    with open(ds_file, "r") as f:
        mhq_ds = json.load(f)

    print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
    print("layers_to_edit: ", layers_to_edit)
    print("ds_file: ", ds_file)
    correct = 0
    for i in range(len(mhq_ds)):
    # for i in range(100):
        if i < continue_from:
            continue
        case = mhq_ds[i]
        request = case["requested_rewrite"]
        generation_prompts = []

        print("\n\n"+4*"***********************************************")
        print(f"Request {i+1}, case_id: {case['case_id']}")


        # Restore fresh copy of model
        try:
            with torch.no_grad():
                for k, v in orig_weights.items():
                    nethook.get_parameter(model, k)[...] = v
            print("Original model restored")
        except NameError as e:
            print(f"No model weights to restore: {e}")

        # Execute rewrite
        model_new, orig_weights = demo_model_editing(
            model, tok, request, generation_prompts, alg_name=ALG_NAME,generate_prompts=False 
            )

        if eval_editing(model, case, rel_prompts, tok, save_dir=edit_hop+"-Eval-" + "-".join(map(str, layers_to_edit))):
            correct += 1

        print(f"Correct: {correct}/{i}")

    print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

In [ ]:
layers_sweep = [[[5]], [[10]], [[15]], [[20]]]
hop_sweep = ["hop1", "hop2"]
for layers in layers_sweep:
    for hop in hop_sweep:
        test_multi_rome(layers, hop, continue_from=100)

Start time:  2025-03-07 00:01:59
layers_to_edit:  [[5]]
ds_file:  dsets/ds_classification/hop1_edits.json


********************************************************************************************************************************************************************************************
Request 101, case_id: 520
No model weights to restore: cannot access local variable 'orig_weights' where it is not associated with a value

###########################################
#                                         #
#  Retrieving ROME-Multi hyperparameters  #
#                                         #
###########################################
Loading from hparams/ROME/EleutherAI_gpt-j-6B.json
ROMEHyperParams(layers=[[5]], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=27, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.fc_out', laye

loss 3.027 = 3.027 + 0.0 + 0.0 avg prob of [ C. S. Forester] 0.04978634789586067
loss 2.443 = 2.391 + 0.032 + 0.02 avg prob of [ C. S. Forester] 0.09582401812076569
loss 1.201 = 1.131 + 0.039 + 0.031 avg prob of [ C. S. Forester] 0.3255510628223419
loss 0.555 = 0.47 + 0.045 + 0.04 avg prob of [ C. S. Forester] 0.6276872754096985
loss 0.357 = 0.26 + 0.048 + 0.049 avg prob of [ C. S. Forester] 0.7717309594154358
loss 0.287 = 0.185 + 0.047 + 0.056 avg prob of [ C. S. Forester] 0.8317849636077881
loss 0.234 = 0.127 + 0.045 + 0.063 avg prob of [ C. S. Forester] 0.8814356923103333
loss 0.201 = 0.087 + 0.045 + 0.069 avg prob of [ C. S. Forester] 0.9168830513954163
loss 0.179 = 0.063 + 0.045 + 0.071 avg prob of [ C. S. Forester] 0.9392799139022827
loss 0.162 = 0.047 + 0.044 + 0.071 avg prob of [ C. S. Forester] 0.9540923833847046
loss 0.149 = 0.035 + 0.042 + 0.071 avg prob of [ C. S. Forester] 0.9651600122451782
loss 0.137 = 0.027 + 0.039 + 0.071 avg prob of [ C. S. Forester] 0.973552525043487

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 3 | Sentence: Babylon 5 was created by C. S. Fore | Token:  5
Rewrite layer is 10
Tying optimization objective to 27
Recording initial value of v*
loss 3.001 = 3.001 + 0.0 + 0.0 avg prob of [ C. S. Forester] 0.05095399171113968
loss 1.47 = 1.42 + 0.015 + 0.035 avg prob of [ C. S. Forester] 0.2453732043504715
loss 0.729 = 0.651 + 0.025 + 0.053 avg prob of [ C. S. Forester] 0.5236315131187439
loss 0.443 = 0.346 + 0.028 + 0.069 avg prob of [ C. S. Forester] 0.7084487080574036
loss 0.327 = 0.216 + 0.029 + 0.083 avg prob of [ C. S. Forester] 0.806686520576477
loss 0.243 = 0.121 + 0.029 + 0.093 avg prob of [ C. S. Forester] 0.8861844539642334
loss 0.192 = 0.074 + 0.026 + 0.093 avg prob of [ C. S. Forester] 0.9291568994522095
loss 0.165 = 0.049 + 0.022 + 0.093 avg prob of [ C. S. Forester] 0.9517984390258789
loss 0.149 = 0.037 + 0.019 + 0.093 avg prob of [ C. S. Forester] 0.9639376997947693
loss 0.139 = 0.02

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 3 | Sentence: Babylon 5 was created by C. S. Fore | Token:  5
Rewrite layer is 15
Tying optimization objective to 27
Recording initial value of v*
loss 3.001 = 3.001 + 0.0 + 0.0 avg prob of [ C. S. Forester] 0.050953950732946396
loss 1.996 = 1.946 + 0.008 + 0.041 avg prob of [ C. S. Forester] 0.14601728320121765
loss 0.728 = 0.642 + 0.024 + 0.063 avg prob of [ C. S. Forester] 0.5284947752952576
loss 0.447 = 0.337 + 0.027 + 0.083 avg prob of [ C. S. Forester] 0.7148936986923218
loss 0.329 = 0.206 + 0.022 + 0.1 avg prob of [ C. S. Forester] 0.8140393495559692
loss 0.235 = 0.119 + 0.015 + 0.101 avg prob of [ C. S. Forester] 0.8883610963821411
loss 0.181 = 0.066 + 0.014 + 0.101 avg prob of [ C. S. Forester] 0.9366195201873779
loss 0.154 = 0.039 + 0.014 + 0.101 avg prob of [ C. S. Forester] 0.9618744850158691
loss 0.14 = 0.026 + 0.013 + 0.101 avg prob of [ C. S. Forester] 0.9747213125228882
loss 0.132 = 0.

  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([16384])
Computing right vector (v)
Lookup index found: 3 | Sentence: Babylon 5 was created by C. S. Fore | Token:  5
Rewrite layer is 20
Tying optimization objective to 27
Recording initial value of v*
loss 3.001 = 3.001 + 0.0 + 0.0 avg prob of [ C. S. Forester] 0.05095395818352699
loss 2.723 = 2.702 + 0.004 + 0.017 avg prob of [ C. S. Forester] 0.0684257298707962
loss 1.729 = 1.688 + 0.014 + 0.026 avg prob of [ C. S. Forester] 0.18861784040927887
loss 1.002 = 0.95 + 0.017 + 0.035 avg prob of [ C. S. Forester] 0.3880111277103424
loss 0.709 = 0.642 + 0.023 + 0.043 avg prob of [ C. S. Forester] 0.5296338796615601
loss 0.24 = 0.165 + 0.026 + 0.049 avg prob of [ C. S. Forester] 0.8483911156654358
loss 0.156 = 0.071 + 0.03 + 0.054 avg prob of [ C. S. Forester] 0.9319979548454285
loss 0.136 = 0.046 + 0.03 + 0.06 avg prob of [ C. S. Forester] 0.9551385641098022
loss 0.124 = 0.033 + 0.026 + 0.065 avg prob of [ C. S. Forester] 0.9673452973365784
loss 0.114 = 0.029

In [20]:
import os
import json

# folder_dir = "hop2-Eval-5-10-15-20"
folder_dir ="both-Eval-[5, 15]-[10, 20]"
folder_dir = "both-Eval-[5]-[5]"
# folder_dir = "hop2-Eval-[10]"
folder_dir = "hop2-Eval-[5]"
# folder_dir = "hop1-Eval-[5, 10, 15, 20]"
# folder_dir = "hop2-Eval-5-10-15-20"
correct = 0
total = len(os.listdir(folder_dir))
# iterate over the json files in the folder
for filename in os.listdir(folder_dir):
    if filename.endswith(".json"):
        with open(os.path.join(folder_dir, filename), "r") as f:
            data = json.load(f)
            if data[-1]["correct"]:
                correct += 1
print(f"{folder_dir} Correctness: {correct}/{total}")

hop2-Eval-[5] Correctness: 14/359


: 

In [17]:
! python3 -m experiments.summarize --dir_name=ROME-Multi --runs=run_003

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/experiments/summarize.py", line 9, in <module>
    from util.globals import *
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/util/__init__.py", line 1, in <module>
    from .logit_lens import LogitLens
  File "/home/jeffhe/KE4MHQ/KE4MHQ/rome/util/logit_lens.py", line 4, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'
